# 01 · Cleaning — messy Titanic

**Dataset:** `data/titanic_raw.csv` — a deliberately dirty Titanic file.
**Covers:** load & first look · missing values · duplicates · dtypes · outliers.
**Time yourself:** ~35 minutes.

Work top to bottom. Say your reasoning out loud as you go — in the real interview the
*why* is worth more than the code. Solutions are in `solutions/01_cleaning_titanic.ipynb`.

In [29]:
import os
if os.path.basename(os.getcwd()) == 'solutions':
    os.chdir('..')

In [30]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
df = pd.read_csv('data/titanic_raw.csv')
df.head()

,PassengerId,Survived,P Class,Name,Sex,Age,sib_sp,Parch,Ticket,Fare,Cabin,Embarked,TicketDate,CabinNote
0,64,0,3,"Skoog, Master. Harald",male,4.0,3,2,347088,$27.90,NaN,s,10/04/1912,NaN
1,678,1,3,"Turja, Miss. Anna Sofia",female,18.0,0,0,4138,$9.84,NaN,s,10/04/1912,NaN
2,141,0,3,"Boulos, Mrs. Joseph (Sultana)",FEMALE,NaN,0,2,2678,$15.25,NaN,C,1912-04-11,NaN
3,54,1,2,"Faunthorpe, Mrs. Lizzie (Elizabeth Anne Wilkin...",Female,29.0,1,0,2926,$26.00,NaN,S,1912-04-12,checked
4,826,0,3,"Flynn, Mr. John",male,NaN,0,0,368323,$6.95,NaN,Q,"April 12, 1912",NaN


---

## Part A — First look

### Q1. How many rows and columns are there? Print the shape, the dtypes and the non-null counts in as few calls as you can.

<details><summary>hint</summary>

`.shape` and `.info()` — `.info()` gives you dtypes and non-null counts at once.

</details>

In [31]:
print(df.shape)
df.info()

(914, 14)
<class 'pandas.DataFrame'>
RangeIndex: 914 entries, 0 to 913
Data columns (total 14 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  914 non-null    int64  
 1   Survived     914 non-null    int64  
 2   P Class      914 non-null    int64  
 3   Name         914 non-null    str    
 4   Sex          914 non-null    str    
 5   Age          733 non-null    float64
 6   sib_sp       914 non-null    int64  
 7    Parch       914 non-null    int64  
 8   Ticket       914 non-null    str    
 9   Fare         914 non-null    str    
 10  Cabin        210 non-null    str    
 11  Embarked     912 non-null    str    
 12  TicketDate   914 non-null    str    
 13  CabinNote    58 non-null     str    
dtypes: float64(1), int64(5), str(8)
memory usage: 100.1 KB


### Q2. Get summary statistics for the numeric columns *and* for the object columns.
Looking at the output: name three columns whose dtype is **wrong** for what they
actually represent, and say what each should be.

<details><summary>hint</summary>

`df.describe()` defaults to numeric only. Pass `include='object'` for the rest.

</details>

In [32]:
display(df.describe())
display(df.describe(include='object'))

# Wrong dtypes:
#   Fare       -> object, because values look like "$27.90"; should be float
#   TicketDate -> object; should be datetime64
#   Sex        -> object with inconsistent casing/whitespace; should be a clean category
# (Survived / P Class are ints but are really categorical — fine to leave as int.)

,PassengerId,Survived,P Class,Age,sib_sp,Parch
count,914.000000,914.000000,914.000000,733.000000,914.000000,914.000000
mean,443.814004,0.387309,2.307440,30.092319,0.518600,0.380744
std,258.993287,0.487402,0.836339,16.918047,1.097734,0.804901
min,1.000000,0.000000,1.000000,-5.000000,0.000000,0.000000
25%,219.250000,0.000000,2.000000,20.000000,0.000000,0.000000
50%,444.500000,0.000000,3.000000,28.000000,0.000000,0.000000
75%,667.750000,1.000000,3.000000,38.000000,1.000000,0.000000
max,891.000000,1.000000,3.000000,200.000000,8.000000,6.000000


C:\Users\Pablo\AppData\Local\Temp\ipykernel_28436\4001823809.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  display(df.describe(include='object'))


,Name,Sex,Ticket,Fare,Cabin,Embarked,TicketDate,CabinNote
count,914,914,914,914,210,912,914,58
unique,891,9,681,236,147,7,9,1
top,"Calderhead, Mr. Edward Pennington",male,1601,$8.05,G6,s,"April 11, 1912",checked
freq,2,131,8,44,4,233,110,58


### Q3. The column names are messy: `'P Class'`, `'sib_sp'`, `' Parch'`, `'PassengerId'`,
`'TicketDate'`. Normalise **all** of them to lowercase snake_case, without typing
each name out.

Watch out for the CamelCase ones — `TicketDate` should become `ticket_date`,
not `ticketdate`.

<details><summary>hint</summary>

`df.columns` is an Index with `.str` accessors, just like a Series. For the CamelCase split, insert an underscore at every lowercase→uppercase boundary with a zero-width regex: `r'(?<=[a-z0-9])(?=[A-Z])'`.

</details>

In [33]:
df.columns = (df.columns
              .str.strip()
              .str.replace(r'(?<=[a-z0-9])(?=[A-Z])', '_', regex=True)  # CamelCase split
              .str.replace(' ', '_', regex=False)
              .str.lower())
print(df.columns.tolist())

['passenger_id', 'survived', 'p_class', 'name', 'sex', 'age', 'sib_sp', 'parch', 'ticket', 'fare', 'cabin', 'embarked', 'ticket_date', 'cabin_note']


---

## Part B — Types & structure

### Q4. Convert `fare` to a proper numeric column. Watch out: values look like `$1,234.56`
and some are blank. Any value that cannot be parsed should become `NaN`, not raise.
How many NaNs did you end up with?

<details><summary>hint</summary>

Strip the `$` and the thousands separator first, then `pd.to_numeric(..., errors='coerce')`. `errors='coerce'` is what turns junk into NaN instead of raising.

</details>

In [34]:
df['fare'] = (df['fare']
              .str.replace('$', '', regex=False)
              .str.replace(',', '', regex=False)
              .str.strip())
df['fare'] = pd.to_numeric(df['fare'], errors='coerce')

print(df['fare'].dtype, '| NaNs:', df['fare'].isna().sum())
df['fare'].describe()

float64 | NaNs: 0


count    914.000000
mean      32.007221
std       49.199110
min        0.000000
25%        7.905000
50%       14.450000
75%       31.000000
max      512.330000
Name: fare, dtype: float64

### Q5. Normalise `sex` and `embarked`: strip whitespace and unify the casing so that
`sex` has exactly 2 unique values and `embarked` has exactly 3 (plus NaN).
Verify with `value_counts(dropna=False)`.

<details><summary>hint</summary>

`.str.strip()` then `.str.lower()` / `.str.upper()`. Always re-check with `value_counts(dropna=False)` — the `dropna=False` is what shows you the NaNs.

</details>

In [35]:
df['sex'] = df['sex'].str.strip().str.lower()
df['embarked'] = df['embarked'].str.strip().str.upper()

print(df['sex'].value_counts(dropna=False))
print(df['embarked'].value_counts(dropna=False))

sex
male      590
female    324
Name: count, dtype: int64
embarked
S      663
C      169
Q       80
NaN      2
Name: count, dtype: int64


### Q6. `ticket_date` is stored as text in **three different formats** (`1912-04-10`,
`10/04/1912`, `April 10, 1912`). Parse it into a real datetime column so that
**zero** values fail to parse.

<details><summary>hint</summary>

`pd.to_datetime(..., format='mixed')` parses each value independently. You also need `dayfirst=True` so `10/04/1912` reads as 10 April, not 4 October.

</details>

In [36]:
df['ticket_date'] = pd.to_datetime(df['ticket_date'], format='mixed', dayfirst=True)

print(df['ticket_date'].dtype, '| failed:', df['ticket_date'].isna().sum())
df['ticket_date'].value_counts().sort_index()

datetime64[us] | failed: 0


ticket_date
1912-04-10    217
1912-04-11    202
1912-04-12    190
1912-10-04    102
1912-11-04    101
1912-12-04    102
Name: count, dtype: int64

### Q7. Cast `sex` and `embarked` to the `category` dtype. Before you do — check the
cardinality to justify it. How much memory did the DataFrame save?

<details><summary>hint</summary>

`.nunique()` for cardinality, `.astype('category')` to convert, `.memory_usage(deep=True)` — `deep=True` matters, otherwise strings aren't counted.

</details>

In [37]:
before = df.memory_usage(deep=True).sum()

for col in ['sex', 'embarked']:
    print(col, '->', df[col].nunique(), 'unique values')
    df[col] = df[col].astype('category')

after = df.memory_usage(deep=True).sum()
print(f'{before/1024:.0f} KB -> {after/1024:.0f} KB')

sex -> 2 unique values
embarked -> 3 unique values
330 KB -> 240 KB


---

## Part C — Duplicates

Do this **before** imputing: imputing first would compute statistics over rows that shouldn't be there.

### Q8. How many fully duplicated rows are there? How many rows share a `passenger_id` with another row? Are those two the same set of rows?

<details><summary>hint</summary>

`.duplicated()` with no args compares every column; `subset=['passenger_id']` compares only the key.

</details>

In [38]:
print('fully duplicated rows :', df.duplicated().sum())
print('duplicated passenger_id:', df.duplicated(subset=['passenger_id']).sum())

# They are not the same set: some rows repeat the id but differ in `age`,
# so they are not caught by the full-row check.

fully duplicated rows : 15
duplicated passenger_id: 23


### Q9. Drop the fully duplicated rows. Then inspect the rows that still share a
`passenger_id` — what is different about them, and what would you do?

<details><summary>hint</summary>

`keep=False` inside `.duplicated()` marks *every* member of a duplicate group, not just the repeats — that's how you see both sides to compare them.

</details>

In [39]:
df = df.drop_duplicates()
print('after dropping exact dupes:', df.shape)

dupe_ids = df.loc[df.duplicated(subset=['passenger_id'], keep=False), 'passenger_id']
display(df[df['passenger_id'].isin(dupe_ids)].sort_values('passenger_id').head(10))

# They are identical except `age` differs by 1 year -- a conflicting record rather than
# a copy. passenger_id is supposed to be a unique key, so one of the two is wrong.
# Without a source of truth, keep one deterministically:
df = df.drop_duplicates(subset=['passenger_id'], keep='first')
print('after resolving key dupes:', df.shape)
print('passenger_id unique?', df['passenger_id'].is_unique)

after dropping exact dupes: (899, 14)


,passenger_id,survived,p_class,name,sex,age,sib_sp,parch,ticket,fare,cabin,embarked,ticket_date,cabin_note
26,38,0,3,"Cann, Mr. Ernest Charles",male,22.0,0,0,A./5. 2152,8.05,NaN,S,1912-11-04,NaN
318,38,0,3,"Cann, Mr. Ernest Charles",male,21.0,0,0,A./5. 2152,8.05,NaN,S,1912-11-04,NaN
152,69,1,3,"Andersson, Miss. Erna Alexandra",female,18.0,4,2,3101281,7.92,NaN,S,1912-12-04,NaN
896,69,1,3,"Andersson, Miss. Erna Alexandra",female,17.0,4,2,3101281,7.92,NaN,S,1912-12-04,NaN
784,170,0,3,"Ling, Mr. Lee",male,28.0,0,0,1601,56.50,NaN,S,1912-04-12,NaN
842,170,0,3,"Ling, Mr. Lee",male,29.0,0,0,1601,56.50,NaN,S,1912-04-12,NaN
639,607,0,3,"Karaic, Mr. Milan",male,30.0,0,0,349246,7.90,NaN,S,1912-10-04,checked
388,607,0,3,"Karaic, Mr. Milan",male,31.0,0,0,349246,7.90,NaN,S,1912-10-04,checked
663,616,1,2,"Herman, Miss. Alice",female,25.0,1,2,220845,65.00,NaN,S,1912-04-11,NaN
907,616,1,2,"Herman, Miss. Alice",female,24.0,1,2,220845,65.00,NaN,S,1912-04-11,NaN


after resolving key dupes: (891, 14)
passenger_id unique? True


---

## Part D — Missing values

### Q10. Build a single table showing, per column, the count **and** the percentage of missing values — sorted worst-first, and only for columns that actually have any.

<details><summary>hint</summary>

`.isnull().sum()` gives counts, `.isnull().mean()` gives the fraction. Build a DataFrame from both and filter.

</details>

In [40]:
miss = pd.DataFrame({
    'n_missing': df.isnull().sum(),
    'pct_missing': (df.isnull().mean() * 100).round(1),
})
miss = miss[miss['n_missing'] > 0].sort_values('pct_missing', ascending=False)
miss

,n_missing,pct_missing
cabin_note,835,93.7
cabin,687,77.1
age,176,19.8
embarked,2,0.2


### Q11. `cabin_note` is ~94% missing. Decide what to do with it and justify it in a comment.

<details><summary>hint</summary>

Above ~60% missing the usual call is to drop — unless the *missingness itself* is the signal, in which case you keep a binary indicator instead.

</details>

In [41]:
# ~94% missing means there is almost nothing to impute from -- any fill would be
# inventing the column rather than recovering it. Drop it.
df = df.drop(columns=['cabin_note'])

### Q12. `cabin` is ~77% missing — but here the missingness may itself be informative.
Instead of dropping it blindly, first **check**: is the survival rate different for
passengers with vs. without a recorded cabin? Then act on what you find.

<details><summary>hint</summary>

`.notna().astype(int)` builds the indicator. Then `groupby` it and take the mean of `survived` — the mean of a 0/1 column is the rate.

</details>

In [42]:
df['has_cabin'] = df['cabin'].notna().astype(int)
print(df.groupby('has_cabin')['survived'].agg(['mean', 'count']))

# Survival is roughly 30% without a cabin vs ~67% with one -- missingness is clearly
# informative (a recorded cabin is a proxy for a paying 1st-class passenger).
# So: keep the indicator, and also salvage the deck letter, then drop the raw column.
df['deck'] = df['cabin'].str[0]
print(df['deck'].value_counts(dropna=False))
df = df.drop(columns=['cabin'])
df

               mean  count
has_cabin                 
0          0.299854    687
1          0.666667    204
deck
NaN    687
C       59
B       47
D       33
E       32
A       15
F       13
G        4
T        1
Name: count, dtype: int64


,passenger_id,survived,p_class,name,sex,age,sib_sp,parch,ticket,fare,embarked,ticket_date,has_cabin,deck
0,64,0,3,"Skoog, Master. Harald",male,4.0,3,2,347088,27.90,S,1912-04-10,0,NaN
1,678,1,3,"Turja, Miss. Anna Sofia",female,18.0,0,0,4138,9.84,S,1912-04-10,0,NaN
2,141,0,3,"Boulos, Mrs. Joseph (Sultana)",female,NaN,0,2,2678,15.25,C,1912-11-04,0,NaN
3,54,1,2,"Faunthorpe, Mrs. Lizzie (Elizabeth Anne Wilkin...",female,29.0,1,0,2926,26.00,S,1912-12-04,0,NaN
4,826,0,3,"Flynn, Mr. John",male,NaN,0,0,368323,6.95,Q,1912-04-12,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
909,580,1,3,"Jussila, Mr. Eiriik",male,32.0,0,0,STON/O 2. 3101286,7.92,S,1912-04-10,0,NaN
910,503,0,3,"O'Sullivan, Miss. Bridget Mary",female,NaN,0,0,330909,7.63,Q,1912-04-12,0,NaN
911,538,1,1,"LeRoy, Miss. Bertha",female,30.0,0,0,PC 17761,106.42,C,1912-04-10,0,NaN
912,197,0,3,"Mernagh, Mr. Robert",male,NaN,0,0,368703,7.75,Q,1912-04-10,0,NaN


### Q13. `age` is ~20% missing. A global median is crude. Impute it with the median age
**of the passenger's own (`p_class`, `sex`) group** instead. Confirm zero NaNs
remain and that the overall distribution barely moved.

<details><summary>hint</summary>

`groupby(...)['age'].transform(...)` returns a Series aligned to the original index — that alignment is why `transform` works here and `apply`/`agg` wouldn't.

</details>

In [43]:
before = df['age'].describe()

df['age'] = df.groupby(['p_class', 'sex'])['age'].transform(lambda s: s.fillna(s.median()))

print('NaNs left:', df['age'].isna().sum())
pd.DataFrame({'before': before, 'after': df['age'].describe()}).round(2)

NaNs left: 0


,before,after
count,715.00,891.00
mean,30.06,29.41
std,16.93,15.44
min,-5.00,-5.00
25%,20.00,21.50
50%,28.00,26.00
75%,38.00,36.00
max,200.00,200.00


### Q14. `embarked` has only a couple of missing values. Fill them with the mode.

<details><summary>hint</summary>

`.mode()` returns a Series (there can be ties) — take `[0]`.

</details>

In [44]:
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])
print(df['embarked'].value_counts(dropna=False))

embarked
S    646
C    168
Q     77
Name: count, dtype: int64


### Q15. `deck` is still mostly missing. Fill it with an explicit `'Unknown'` category rather than a mode — why is that the better call here?

<details><summary>hint</summary>

Think about what mode-filling 77% of a column does to that column's distribution.

</details>

In [45]:
df['deck'] = df['deck'].fillna('Unknown')

# Mode-filling ~77% of a column would stamp one deck letter onto 3 of every 4 rows and
# invent a pattern that isn't in the data. 'Unknown' is honest: it keeps the rows usable
# while letting the model learn that "no deck recorded" is its own group.
print(df['deck'].value_counts())

deck
Unknown    687
C           59
B           47
D           33
E           32
A           15
F           13
G            4
T            1
Name: count, dtype: int64


---

## Part E — Outliers

### Q16. Some `age` values are physically impossible. Find them. How many rows, and what
are the values?

<details><summary>hint</summary>

Ages below 0 or above ~100 aren't outliers, they're data-entry errors.

</details>

In [46]:
bad = df[(df['age'] < 0) | (df['age'] > 100)]
print(len(bad), 'impossible ages:', sorted(bad['age'].unique()))

5 impossible ages: [np.float64(-5.0), np.float64(-1.0), np.float64(123.0), np.float64(150.0), np.float64(200.0)]


### Q17. Fix the impossible ages. Don't just delete the rows — the other columns in them are
fine. Treat the bad value as missing and re-impute it the same way as before.

<details><summary>hint</summary>

Dropping the row throws away 10 good columns to fix 1 bad cell. Set it to NaN and route it back through your imputation.

</details>

In [47]:
df.loc[(df['age'] < 0) | (df['age'] > 100), 'age'] = np.nan
df['age'] = df.groupby(['p_class', 'sex'])['age'].transform(lambda s: s.fillna(s.median()))

print('NaNs:', df['age'].isna().sum(), '| range:', df['age'].min(), '-', df['age'].max())

NaNs: 0 | range: 0.0 - 81.0


### Q18. Count the `fare` outliers using the 1.5×IQR rule. Then look at the biggest ones —
are they errors, or real?

<details><summary>hint</summary>

`.quantile([0.25, 0.75])` returns both at once and unpacks into two variables.

</details>

In [48]:
q1, q3 = df['fare'].quantile([0.25, 0.75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr

outliers = df[(df['fare'] < lower) | (df['fare'] > upper)]
print(f'bounds: {lower:.2f} to {upper:.2f} | {len(outliers)} outliers '
      f'({len(outliers)/len(df):.1%})')
display(outliers.nlargest(5, 'fare')[['p_class', 'name', 'fare', 'survived']])

# They are real: the top fares are all 1st-class passengers. This is a genuinely
# right-skewed distribution, not corruption -- do not delete them.

bounds: -26.72 to 65.63 | 116 outliers (13.0%)


,p_class,name,fare,survived
239,1,"Lesurer, Mr. Gustave J",512.33,1
494,1,"Cardeza, Mr. Thomas Drake Martinez",512.33,1
529,1,"Ward, Miss. Anna",512.33,1
384,1,"Fortune, Mr. Charles Alexander",263.00,0
614,1,"Fortune, Miss. Mabel Helen",263.00,1


### Q19. You decided the extreme fares are real. Handle them two ways and keep both columns:
(a) a winsorized `fare_capped` clipped at the 1st/99th percentile, and
(b) a `fare_log` using a log transform that tolerates the zero fares.
Compare the skew of all three.

<details><summary>hint</summary>

`.clip(lo, hi)` winsorizes. `np.log1p` = log(1+x), which is defined at x=0 — plain `np.log` would give you -inf on the free tickets.

</details>

In [ ]:
lo, hi = df['fare'].quantile([0.01, 0.99])
df['fare_capped'] = df['fare'].clip(lo, hi)
df['fare_log'] = np.log1p(df['fare'])          # log1p handles fare == 0 safely

print(df[['fare', 'fare_capped', 'fare_log']].skew().round(2))

---

## Part F — Wrap up

### Q20. Final check, then save. Assert that: there are no missing values left, `passenger_id`
is unique, and `age`/`fare` are within sane ranges. Save to
`data/titanic_cleaned_by_me.csv`.

<details><summary>hint</summary>

`assert` statements are a nice thing to write in an interview — they show you think about verifying your own work.

</details>

In [ ]:
assert df.isnull().sum().sum() == 0, df.isnull().sum()[lambda s: s > 0]
assert df['passenger_id'].is_unique
assert df['age'].between(0, 100).all()
assert (df['fare'] >= 0).all()

df.to_csv('data/titanic_cleaned_by_me.csv', index=False)
print('saved:', df.shape)
df.head()

---

## Debrief — what an interviewer is checking

- **Split before you fit.** Every imputation here was computed on the whole file. That is
  fine for an exploratory clean, but the moment a fill value comes from a *statistic*
  (median, mode, group mean), it must be learned on train only. Notebook 03 redoes this
  properly inside a `Pipeline`.
- **You checked before you dropped.** The `cabin` question is the trap: 77% missing looks
  like an easy drop, but the missingness carried most of the signal.
- **Errors vs. outliers.** A negative age is a bug; a £512 fare is a fact. Different fixes.
- **Order matters.** Duplicates before imputation; type fixes before anything numeric.